# Inimene-tsüklis: eeltoimingu väravad, riskitasemed ja auditeerimislogi

Selle õppetüki README tutvustab Inimest-tsüklis lühikese lõiguga, mis küsib kasutajalt `HEAKSKIITA` või `HÜVASTI` pärast seda, kui agent on juba vastuse loonud. See muster on hea lähtepunkt, kuid tootmises kasutatakse HITL-i tavaliselt koos kolme lisatükiga:

1. **eeltoimingu värav**, mis käivitub **enne** agenti riskantse sammu täitmist, et hoida kulud, tagasipöördumatus ja latentsus kontrolli all.
2. **riskitasemed**, nii et madala riskiga toimingud täidetakse automaatselt, keskmise riskiga toimingud heaks kiidetakse partiidena ja ainult kõrge riskiga toimingud jäävad inimtõrjele.
3. **auditeerimislogi ja redigeerimis-tsükkel**, nii et iga värava otsus salvestatakse JSONL formaadis ja tagasilükkamine käivitab agenti struktuurse põhjendusega, mitte lihtsalt käsklust `Redigeerimine...` printida.

See märkmik ehitab kõik need üles samade põhielementide baasil nagu `06-system-message-framework.ipynb`. See töötab lõpuni `DEMO_MODE = True` (interaktiivse sisendi vajadus puudub) või tõeliste `input()` viipadega, kui `DEMO_MODE = False`. Märkus: DEMO_MODE puhul on kolmanda eesmärgi kordus skriptitud, nii et tsükli mehhaanikad on nähtavad lõpuni. Tõeline redigeerimispõhine ümberklassifitseerimine nõuab `DEMO_MODE = False` ja operaatorit.

**Välja jäetud valdkonnad (kaetud teistes õppetundides):** autentimine ja juurdepääsu kontroll (Õppetund 06 README oht 2), tööriistakõne vahendustarkvara (Õppetund 14 MAF põhjalik analüüs), mitme-agendi vaidlusmustrid.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Muster 1: Eeltegevuse värav

README HITL-koodilõik kutsub esmalt agendi, seejärel palub kasutajal väljundi heaks kiita. See on **järeltegevuse** voog. Agent on juba käivitanud, seega LLM-i kõne kulu on juba tasutud ja kõik kõrvaltoimed (saadetud e-kiri, kirjutatud andmebaasi rida, postitatud kommentaar) on juba toimunud.

**Eeltegevuse** voog lisab värava enne, kui agent käivitab riskantse sammu. Agent teeb ettepaneku teo kohta, värav otsustab, kas tegevust teostada, ja ainult heakskiidul toimub kõrvaltoime.

| Aspekt | Järeltegevuse heakskiit (README koodilõik) | Eeltegevuse värav (see märkmik) |
|---|---|---|
| Millal käib heakskiit? | Pärast seda, kui agent on väljundi tootnud | Enne igasugust kõrvaltoimet |
| LLM-i kulu tagasilükkamisel | Juba tasutud | Tasutud ainult ettepaneku, mitte teo eest |
| Tagasipöördumatud kõrvaltoimed | Võimalikud (tegevus juba toimunud) | Takistatud |
| Auditeerimise selgus | Heakskiit on print-lause | Heakskiit on ajatempli, teost ja põhjusega JSON-kirje |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Muster 2: Riski tasemed

Mitte iga tegevus ei vaja inimkinnitust. Avaliku API vastu tehtav ainult lugemine on erineva tähtsusega kui kliendile e-kirja saatmine. Mõlema sama kohtlemine raiskab operaatori tähelepanu ja aeglustab agendi tööd.

Lihtne 3-tasemeline mudel:

| Tase | Näited | Kinnituse voog |
|---|---|---|
| `madal` (ainult lugemine) | Otsi teadmistebaasist, vaata lennuvõimalusi, saada avalik veebileht | Täida automaatselt, logitakse auditi jaoks |
| `keskmine` (odav muutmine) | Vahemälu tulemus, loendurit suurendada, meeldetuletust ajastada | Täida automaatselt, kuid kogutud päevaseks ülevaateks |
| `kõrge` (välise suunaga või pöördumatu) | Saada e-kiri, võta kaartelt tasu, saada avalikku kanalisse postitus | Peata kuni inimkinnituseni |

See on üks tasemejaotus. Tootmiskeskkonnad kasutavad sageli täpsemaid tasemeid (näiteks AWS IAM õiguste tasemed, rollipõhised ligipääsu tasemed). Alljärgnev 3-tasemeline versioon on väikseim kasulik versioon agendile, mis segab ainult lugemisega ja kõrvalmõjuga tegevusi.

Allolev klassifikaator kasutab märksõnade heuristikat, et demo oleks määratletud ja odav. Tootmiskeskkonnas vahetaksite selle välja õpitud klassifikaatori või poliitika mootoriga.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Muster 3: Auditilogii ja paranduste tsükkel

`print("Vastus heaks kiidetud.")` ei ole auditilogii. Usalduse tagamiseks tuleks iga väravotsus salvestada struktureeritud sündmusena, mida saate hiljem pärida, taasesitada või lisada intsidentide ülevaatele.

Kaks osa:

1. **Ainult lisamisega JSONL.** Üks rida iga otsuse kohta, koos ajatempli, toimingu, tasandi, otsuse ja põhjusega. Lihtne grepida, lihtne hiljem tõelisse logipoodi saata.
2. **Paranduste tsükkel tagasilükkamisel.** Kui värav tagastab `deny` (keeldu), küsib agent endalt uuesti, tuues kontekstis välja tagasilükkamise põhjuse, et järgmine ettepanek saaks probleemist hoiduda.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Täiendavad ressursid

Mitmed teised avalikud projektid rakendavad nende HITL mustrite erinevaid varieerumisi. Võrrelge lähenemisi, et leida oma süsteemile sobivaim:

- **LangChain** inimese järel kontrolli tööriistakihid ([dokumentatsioon](https://python.langchain.com/docs/integrations/tools/human_tools)): otse kasutatavad tööriistakihid, mis peatavad täitmise inimsisendi ootamiseks.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentatsioon](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ seda ümber struktureeris): kasutab agendi rolli, mis esindab inimest mitme-agendi vestlustes.
- **Microsoft Agent Framework (MAF)** funktsiooni kutsumise vahendustarkvara ([dokumentatsioon](https://learn.microsoft.com/agent-framework/)): vahendustarkvara, mis jookseb iga tööriista/funktsiooni kõne ümber, sobib loogika kontrolliks ja kinnitusevoogude haldamiseks.

Iga projekt käsitleb kolme alammustrit erinevalt: LangChain pakendab need tööriistadena, AutoGen kasutab agendi rolli ja Microsoft Agent Framework kasutab funktsiooni kutsumise vahendustarkvara. Lugege üks või kaks rakendust algusest lõpuni enne oma agendi disaini valimist.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
